## OSRM Network Query

Key files needed: 

| File | Used for |
|---|---|
| output/hdb_blocks.gpkg | Residential points - my origins | 
| data/ChildCareServices_2021.geojson| Residential points - my destinations| 
| data/MasterPlanSubZones_2019.geojson | Study area |

Key Dependencies:

| Library | Purpose |
|---|---|
| `xlogit` | Model estimation |
| `pandas` | Tabular data manipulation |
| `matplotlib` | QA scatter plots |
| `pickle` | Save and export models |
| `numpy` | Data processing |
| `statsmodels` | |
| `seaborn` | Graphs for checking correlation between model variables|
| `folium` | Visualisation of routes|


The extracted OSRM Singapore files can also be directly downloaded here. 
Data for running OSRM via Docker:
https://www.dropbox.com/scl/fo/de4tvoq346ylpexiz2k2s/AAKRUqoSCLjPv95NMsHSMMM?rlkey=sddbtns26x7skqf0d8nh98yak&st=3ytafdb2&dl=0

But, if you prefer to extract the data yourself, please follow the manual extraction instructions:


--- 

### Manual Extraction

1. First, please obtain the OSRM files through running osrm-extract in Docker on the singapore_only osm.pbf file in your Shell/Terminal:

OSRM uses Lua profiles (such as car.lua, bicycle.lua, or foot.lua) to determine which roads are passable for a specific transport mode. Use the osrm-extract tool to parse the PBF data.

```bash
docker run -t -v "${PWD}:/data" osrm/osrm-backend osrm-extract -p /opt/car.lua /data/singapore.osm.pbf
```

(Replace with *walk* for the walking profiles)

2. Next, prepare the data for the Multi-Level Dijkstra (MLD) routing algorithm. This requires partitioning the map into cells and calculating routing weights.

#### Partition the graph into cells
```bash
docker run -t -v "${PWD}:/data" osrm/osrm-backend osrm-partition /data/singapore-latest.osrm`
```

#### Customize the cells with routing weights
```bash
docker run -t -v "${PWD}:/data" osrm/osrm-backend osrm-customize /data/singapore-latest.osrm`
```

### Local OSRM routing servers

Accessibility costs (\(C_{i,j}^{m}\)) are computed via local OSRM instances, one per mode. Each assumes the corresponding `.osrm` graph has already been built (`osrm-extract` → `osrm-partition` → `osrm-customize`) into the mounted data folder.

| Mode    | Port | Data folder       |
|---------|------|--------------------|
| Walking | 5001 | `OSRMData_Walk`    |
| Driving | 5002 | `OSRMData_Drive`   |



```bash
# Walking — port 5001
docker run --volume="<your_path>/OSRMData_Walk:/data" -p 5001:5000 -d osrm/osrm-backend
```

```bash
# Driving — port 5002
docker run --volume="<your_path>/OSRMData_Drive:/data" -p 5002:5000 -d osrm/osrm-backend
\`\`\`

# ─────────────────────────────────────────────

### Notes for BT4015 Analysis
How to calculate timings between each origin + destination?
###
Neighbourhood - its closest childcare centre 

In [9]:
#origin - destination matrix

import requests
import numpy as np
import pandas as pd
import geopandas as gpd

def get_osrm_matrix(origins, destinations, mode="foot", metric="duration"):
    """Return an OSRM matrix: duration in minutes or distance in metres."""
    if metric not in ("duration", "distance"):
        raise ValueError("metric must be 'duration' or 'distance'")

    coords = origins + destinations
    coords_str = ";".join(f"{lon},{lat}" for lon, lat in coords)

    url = f"http://localhost:5002/table/v1/{mode}/{coords_str}"
    params = {
        "sources": ";".join(map(str, range(len(origins)))),
        "destinations": ";".join(map(str, range(len(origins), len(coords)))),
        "annotations": metric,
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()

    if data.get("code") != "Ok":
        raise RuntimeError(f"OSRM error: {data.get('message', data)}")

    key = "durations" if metric == "duration" else "distances"
    matrix = np.array(
        [
            [value if value is not None else np.inf for value in row]
            for row in data[key]
        ],
        dtype=float,
    )

    return matrix / 60 if metric == "duration" else matrix


def chunk_list(coords, chunk_size):
    """Yield coordinate batches of at most chunk_size."""
    for start in range(0, len(coords), chunk_size):
        yield coords[start:start + chunk_size]

### Calculating timings for walk matrix

Matrix jsons are saved in a .gitignore folder /large_data - file sizes should be around ~150 and ~200 mb respectively

In [10]:
#files

hdbs = gpd.read_file("../large_data/hdb_points_TEST.geojson")
ccs = gpd.read_file("../data/ChildCareServices_2021.geojson")

print(f"Loaded {len(hdbs)} HDB points and {len(ccs)} childcare services.")

Loaded 10741 HDB points and 1925 childcare services.


In [11]:
from tqdm.auto import tqdm

gdf_origins = hdbs.copy()
gdf_destinations = ccs.copy()

gdf_origins = gdf_origins.to_crs(epsg=4326) #wgs84, osrm only works in global coordinates
gdf_destinations = gdf_destinations.to_crs(epsg=4326)

# OSRM coordinate order: (longitude, latitude)
coords_origins = list(zip(
    gdf_origins.geometry.x,
    gdf_origins.geometry.y
))
coords_destinations = list(zip(
    gdf_destinations.geometry.x,
    gdf_destinations.geometry.y
))

print(coords_origins[:3])
#should be around (103, 1.3)


#maximum chunk size for OSRM requests
max_size = 100

# crear una matriz completa llena de infinitos, de tamaño (num_origenes x num_destinos)
matrix_time = np.full((len(coords_origins), len(coords_destinations)), np.inf)
matrix_dist = np.full((len(coords_origins), len(coords_destinations)), np.inf)

# dividir orígenes y destinos en sub-listas
origen_chunks = list(chunk_list(coords_origins, max_size))
destino_chunks = list(chunk_list(coords_destinations, max_size))

num_origins = len(coords_origins)
num_destinations = len(coords_destinations)
total_queries = 2 * len(origen_chunks) * len(destino_chunks)
print(f"Origins: {num_origins:,} | Destinations: {num_destinations:,}")
print(f"Origin-destination pairs: {num_origins * num_destinations:,} | OSRM queries: {total_queries:,}")

# Each batch makes two requests: duration and distance.
with tqdm(total=total_queries, desc="OSRM queries", unit="query") as progress:
    for i, sub_origins in enumerate(origen_chunks):
        for j, sub_destinations in enumerate(destino_chunks):
            ini_i = i * max_size
            fin_i = ini_i + len(sub_origins)
            ini_j = j * max_size
            fin_j = ini_j + len(sub_destinations)
            progress.set_postfix_str(
                f"origins {ini_i + 1:,}-{fin_i:,}/{num_origins:,}, "
                f"destinations {ini_j + 1:,}-{fin_j:,}/{num_destinations:,}"
            )

            sub_matriz_t = get_osrm_matrix(sub_origins, sub_destinations, mode='foot', metric='duration')
            matrix_time[ini_i:fin_i, ini_j:fin_j] = sub_matriz_t
            progress.update(1)

            sub_matriz_d = get_osrm_matrix(sub_origins, sub_destinations, mode='foot', metric='distance')
            matrix_dist[ini_i:fin_i, ini_j:fin_j] = sub_matriz_d
            progress.update(1)

# convertir la matriz (numpy array) en un DataFrame de pandas, con index de orígenes y columns de destinos
walk_matrix_time = pd.DataFrame(matrix_time, index=gdf_origins.index, columns=gdf_destinations.index)
walk_matrix_dist = pd.DataFrame(matrix_dist, index=gdf_origins.index, columns=gdf_destinations.index)

# export matrix to json for use 
walk_matrix_time.to_json("../large_data/walk_matrix_time.json", orient="split")
walk_matrix_dist.to_json("../large_data/walk_matrix_dist.json", orient="split")

[(103.74684200757073, 1.3413230558229585), (103.76036105079291, 1.3856340613434799), (103.79346353096793, 1.307620048585763)]
Origins: 10,741 | Destinations: 1,925
Origin-destination pairs: 20,676,425 | OSRM queries: 4,320


OSRM queries: 100%|██████████| 4320/4320 [07:37<00:00,  9.44query/s, origins 10,701-10,741/10,741, destinations 1,901-1,925/1,925]


# ─────────────────────────────────────────────

## Route Visualisation - Folium
Visualising shortest paths from each hdb centroid to each childhood centre

In [12]:
# Map a reproducible sample of HDBs and their nearest childcare by walking time.
import folium
from folium.plugins import MarkerCluster
from html import escape
from tqdm.auto import tqdm

MAX_ORIGINS = 100  # Set to None to show every origin (many route requests).
OSRM_WALK_URL = "http://localhost:5001"

if not coords_origins or not coords_destinations:
    raise ValueError("Run the matrix cell first with non-empty origins and destinations.")
if (walk_matrix_time.shape != (len(coords_origins), len(coords_destinations))
        or not walk_matrix_time.index.equals(gdf_origins.index)
        or not walk_matrix_time.columns.equals(gdf_destinations.index)):
    raise ValueError("The walking matrix does not match the current points. Rerun the matrix cell.")

origin_positions = np.arange(len(coords_origins))
if MAX_ORIGINS is not None and len(origin_positions) > MAX_ORIGINS:
    origin_positions = np.sort(np.random.default_rng(42).choice(
        origin_positions, size=MAX_ORIGINS, replace=False
    ))

# OSRM coordinates are (longitude, latitude); Folium uses (latitude, longitude).
map_center = [np.mean([lat for lon, lat in coords_origins]),
              np.mean([lon for lon, lat in coords_origins])]
m = folium.Map(location=map_center, zoom_start=12, tiles="OpenStreetMap")
origin_layer = MarkerCluster(name="HDB origins (displayed sample)").add_to(m)
destination_layer = MarkerCluster(name="Childcare centres (all)").add_to(m)
route_layer = folium.FeatureGroup(name="Routes to nearest childcare by time").add_to(m)

for j, (lon, lat) in enumerate(coords_destinations):
    label = escape(str(gdf_destinations.index[j]))
    folium.Marker(
        [lat, lon], popup=f"Childcare index: {label}",
        icon=folium.Icon(color="green", icon="info-sign")
    ).add_to(destination_layer)

print(f"Displaying {len(origin_positions):,}/{len(coords_origins):,} HDB origins "
      f"and {len(coords_destinations):,} childcare destinations.")
routes_drawn = 0
no_finite_time = 0
route_errors = []
with requests.Session() as session:
    for i in tqdm(origin_positions, desc="Loading walking routes", unit="origin"):
        origin = coords_origins[i]
        origin_label = escape(str(gdf_origins.index[i]))
        times = walk_matrix_time.iloc[i].to_numpy(dtype=float)
        finite_positions = np.flatnonzero(np.isfinite(times))
        popup = f"HDB index: {origin_label}"
        if len(finite_positions):
            j = int(finite_positions[np.argmin(times[finite_positions])])
            destination_label = escape(str(gdf_destinations.index[j]))
            popup += (f"<br>Nearest childcare index: {destination_label}"
                      f"<br>Matrix walking time: {times[j]:.1f} min")
            destination = coords_destinations[j]
            coordinates = f"{origin[0]},{origin[1]};{destination[0]},{destination[1]}"
            try:
                response = session.get(
                    f"{OSRM_WALK_URL}/route/v1/foot/{coordinates}",
                    params={"overview": "full", "geometries": "geojson"},
                    timeout=30,
                )
                response.raise_for_status()
                data = response.json()
                if data.get("code") != "Ok" or not data.get("routes"):
                    raise ValueError(data.get("message", data.get("code", "No route returned")))
                route = data["routes"][0]
                route_popup = (popup + f"<br>Route distance: {route['distance'] / 1000:.2f} km"
                               f"<br>Route time: {route['duration'] / 60:.1f} min")
                folium.PolyLine(
                    [(lat, lon) for lon, lat in route["geometry"]["coordinates"]],
                    color="blue", weight=3, opacity=0.65,
                    tooltip=f"HDB {origin_label} → childcare {destination_label}",
                    popup=route_popup,
                ).add_to(route_layer)
                routes_drawn += 1
            except (requests.RequestException, ValueError, KeyError, IndexError) as exc:
                route_errors.append(f"HDB {gdf_origins.index[i]}: {exc}")
                popup += "<br>Route geometry unavailable"
        else:
            no_finite_time += 1
            popup += "<br>No finite travel times in matrix (unreachable or failed query)"

        folium.Marker(
            [origin[1], origin[0]], popup=popup,
            icon=folium.Icon(color="blue", icon="home")
        ).add_to(origin_layer)

folium.LayerControl().add_to(m)
m.save("routes_map.html")
print(f"Saved routes_map.html: {routes_drawn:,} routes; "
      f"{no_finite_time:,} origins without finite times; {len(route_errors):,} route errors.")
for error in route_errors[:5]:
    print(error)

m


Displaying 100/10,741 HDB origins and 1,925 childcare destinations.


Loading walking routes: 100%|██████████| 100/100 [00:01<00:00, 72.72origin/s]


Saved routes_map.html: 0 routes; 0 origins without finite times; 100 route errors.
HDB 468: HTTPConnectionPool(host='localhost', port=5001): Max retries exceeded with url: /route/v1/foot/103.9585810342282,1.3593320014437038;103.95836736301752,1.358252250178413?overview=full&geometries=geojson (Caused by NewConnectionError("HTTPConnection(host='localhost', port=5001): Failed to establish a new connection: [Errno 61] Connection refused"))
HDB 681: HTTPConnectionPool(host='localhost', port=5001): Max retries exceeded with url: /route/v1/foot/103.85457088085174,1.3622866669899547;103.85365523730248,1.3644533493932327?overview=full&geometries=geojson (Caused by NewConnectionError("HTTPConnection(host='localhost', port=5001): Failed to establish a new connection: [Errno 61] Connection refused"))
HDB 725: HTTPConnectionPool(host='localhost', port=5001): Max retries exceeded with url: /route/v1/foot/103.90130937003732,1.3969442584143215;103.90221403389272,1.3955216697579373?overview=full&geome